# Solution PromoPro — entraînement complet (jeu ResPlan entier)

Carnet autonome : **Exécution → Tout exécuter**, rien d'autre à faire. Il télécharge ResPlan, rastérise les 13 000 plans d'entraînement, entraîne un U-Net ResNet-34 (fond, mur, porte, fenêtre), exporte le modèle en ONNX et le télécharge.

Si Colab coupe la session : rouvrez le carnet et refaites **Tout exécuter**. Les images rastérisées, les points de reprise et le meilleur modèle sont sur votre Google Drive (`MyDrive/promopro-ia`) : tout reprend où ça s'était arrêté.

Réglage à faire une seule fois avant de lancer : **Exécution → Modifier le type d'exécution → GPU (T4)**.

Attribution (licence CC BY 4.0 du jeu de données) : Abouagour & Garyfallidis, « ResPlan: A Large-Scale Vector-Graph Dataset of 17,000 Residential Floor Plans », 2025 — https://github.com/m-agour/ResPlan

In [ ]:
# 1. Dépendances (les avertissements de pip sur protobuf sont sans effet ici)
%pip -q install segmentation-models-pytorch shapely opencv-python-headless onnx onnxscript onnxruntime 'protobuf<6'
import torch, os, time
print('GPU :', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN — activez le GPU : Exécution → Modifier le type d\'exécution')

In [ ]:
# 2. Google Drive (reprise automatique) — repli local si le montage échoue, avec avertissement
DOSSIER = '/content/promopro-ia'
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DOSSIER = '/content/drive/MyDrive/promopro-ia'
    print('Drive monté : reprise automatique active, dossier', DOSSIER)
except Exception as e:
    print('ATTENTION : Drive non monté (', e, '). L\'entraînement continue en local SANS reprise après coupure.')
    print('Pour la reprise : icône dossier à gauche → Monter Drive, puis Exécution → Tout exécuter.')
os.makedirs(DOSSIER, exist_ok=True)
RASTER_ZIP = f'{DOSSIER}/raster.zip'; REPRISE = f'{DOSSIER}/reprise.pt'; MEILLEUR = f'{DOSSIER}/meilleur.pt'; ONNX = f'{DOSSIER}/promopro-plan3d.onnx'

In [ ]:
# 3. Jeu de données ResPlan (téléchargement automatique, Git LFS si nécessaire)
import glob, zipfile, pickle, subprocess
if not os.path.exists('/content/ResPlan'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/m-agour/ResPlan.git', '/content/ResPlan'], check=True)
zips = glob.glob('/content/ResPlan/**/ResPlan*.zip', recursive=True)
if zips and os.path.getsize(zips[0]) < 1_000_000:  # pointeur Git LFS : récupérer le vrai fichier
    subprocess.run('apt-get -qq install -y git-lfs > /dev/null && cd /content/ResPlan && git lfs install -q && git lfs pull', shell=True)
zips = glob.glob('/content/ResPlan/**/ResPlan*.zip', recursive=True)
print('archive :', zips, [os.path.getsize(z) // 1_000_000 for z in zips], 'Mo')
if zips and not os.path.exists('/content/resplan_data'):
    with zipfile.ZipFile(zips[0]) as z: z.extractall('/content/resplan_data')
pkls = glob.glob('/content/resplan_data/**/*.pkl', recursive=True) + glob.glob('/content/ResPlan/**/*.pkl', recursive=True)
assert pkls, 'Aucun fichier .pkl ResPlan trouvé : le dépôt a peut-être changé de structure.'
donnees = pickle.load(open(pkls[0], 'rb'))
plans = donnees if isinstance(donnees, list) else list(donnees.values())
print('plans chargés :', len(plans), '| clés du premier :', list(plans[0].keys()) if isinstance(plans[0], dict) else type(plans[0]))

In [ ]:
# 4. Rastérisation du jeu complet (mise en cache sur Drive : faite une seule fois)
import numpy as np, cv2, random, shutil
from shapely.geometry import shape, Polygon, MultiPolygon
TAILLE = 512
SORTIE = '/content/resplan_raster'

def polygones(plan, cle):
    v = plan.get(cle) if isinstance(plan, dict) else None
    if v is None: return []
    out = []
    for g in (v if isinstance(v, list) else [v]):
        if isinstance(g, dict) and 'type' in g: g = shape(g)
        if isinstance(g, (Polygon, MultiPolygon)):
            for p in (g.geoms if isinstance(g, MultiPolygon) else [g]): out.append(np.array(p.exterior.coords))
        elif isinstance(g, dict) and 'polygon' in g: out.append(np.array(g['polygon']))
        else:
            try: out.append(np.array(g, dtype=float).reshape(-1, 2))
            except Exception: pass
    return out

CLES = {'murs': ['walls', 'wall'], 'portes': ['doors', 'door'], 'fenetres': ['windows', 'window'], 'pieces': ['rooms', 'room', 'spaces']}
def premiere_cle(plan, candidats):
    return next((c for c in candidats if isinstance(plan, dict) and c in plan), None)

def rasteriser(plan):
    tout = [p for cle in CLES.values() for k in [premiere_cle(plan, cle)] if k for p in polygones(plan, k)]
    if not tout: return None
    pts = np.concatenate(tout); mn, mx = pts.min(0), pts.max(0)
    etendue = max(mx - mn)
    if etendue <= 0: return None
    echelle = (TAILLE - 40) / etendue; decal = 20 + ((TAILLE - 40) - (mx - mn) * echelle) / 2  # letterbox : proportions conservées
    def px(p): return ((p - mn) * echelle + decal).astype(np.int32)
    image = np.full((TAILLE, TAILLE, 3), 255, np.uint8); masque = np.zeros((TAILLE, TAILLE), np.uint8)
    k = premiere_cle(plan, CLES['pieces'])
    for p in (polygones(plan, k) if k else []): cv2.fillPoly(image, [px(p)], (247, 245, 240))
    for nom, classe, couleur in [('murs', 1, (25, 25, 25)), ('fenetres', 3, (120, 160, 220)), ('portes', 2, (200, 140, 60))]:
        k = premiere_cle(plan, CLES[nom])
        for p in (polygones(plan, k) if k else []):
            cv2.fillPoly(masque, [px(p)], classe); cv2.fillPoly(image, [px(p)], couleur)
    # Augmentations « vrai plan » : texte parasite, cotes, rotation légère, flou, compression JPEG
    for _ in range(random.randint(2, 8)):
        cv2.putText(image, random.choice(['Salon', 'Cuisine', '3,45', 'Ch. 1', 'SDB', '12 m²', 'WC', 'Terrasse', '4.20 x 3.10']), (random.randint(30, TAILLE - 120), random.randint(30, TAILLE - 30)), cv2.FONT_HERSHEY_SIMPLEX, random.uniform(0.35, 0.7), (40, 40, 40), 1, cv2.LINE_AA)
    angle = random.uniform(-3, 3); M = cv2.getRotationMatrix2D((TAILLE / 2, TAILLE / 2), angle, 1)
    image = cv2.warpAffine(image, M, (TAILLE, TAILLE), borderValue=(255, 255, 255)); masque = cv2.warpAffine(masque, M, (TAILLE, TAILLE), flags=cv2.INTER_NEAREST, borderValue=0)
    if random.random() < 0.5: image = cv2.GaussianBlur(image, (3, 3), 0)
    if random.random() < 0.5:
        ok, enc = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, random.randint(35, 80)]); image = cv2.imdecode(enc, 1)
    return image, masque

if os.path.exists(RASTER_ZIP) and not os.path.isdir(SORTIE):
    print('images rastérisées trouvées sur Drive : décompression…')
    with zipfile.ZipFile(RASTER_ZIP) as z: z.extractall('/content')
if not os.path.isdir(SORTIE) or len(glob.glob(f'{SORTIE}/*_m.png')) < 1000:
    os.makedirs(SORTIE, exist_ok=True); random.seed(0); random.shuffle(plans); n = 0; debut = time.time()
    for i, plan in enumerate(plans):
        r = rasteriser(plan)
        if r is None: continue
        cv2.imwrite(f'{SORTIE}/{i:05d}.png', r[0]); cv2.imwrite(f'{SORTIE}/{i:05d}_m.png', r[1]); n += 1
        if n % 2000 == 0: print(f'  {n} images… ({int(time.time() - debut)} s)')
    assert n > 1000, 'Trop peu de géométrie lue : la structure de ResPlan ne correspond pas aux clés attendues.'
    shutil.make_archive(RASTER_ZIP[:-4], 'zip', '/content', 'resplan_raster')
    print('rastérisation terminée et mise en cache sur Drive')
print('images disponibles :', len(glob.glob(f'{SORTIE}/*_m.png')))

In [ ]:
# 5. Entraînement complet avec reprise automatique
import torch.nn as nn, segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
EPOQUES = 15; LOT = 8; LR = 3e-4
fichiers = sorted(f for f in glob.glob(f'{SORTIE}/*.png') if not f.endswith('_m.png')); coupe = int(len(fichiers) * 0.92)
moy = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); ecart = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
class Jeu(Dataset):
    def __init__(s, f): s.f = f
    def __len__(s): return len(s.f)
    def __getitem__(s, i):
        img = cv2.cvtColor(cv2.imread(s.f[i]), cv2.COLOR_BGR2RGB); m = cv2.imread(s.f[i][:-4] + '_m.png', 0)
        x = (torch.from_numpy(img).permute(2, 0, 1).float() / 255 - moy) / ecart
        return x, torch.from_numpy(m).long()
dl_train = DataLoader(Jeu(fichiers[:coupe]), batch_size=LOT, shuffle=True, num_workers=2, drop_last=True)
dl_val = DataLoader(Jeu(fichiers[coupe:]), batch_size=LOT, num_workers=2)
app = 'cuda' if torch.cuda.is_available() else 'cpu'
modele = smp.Unet('resnet34', encoder_weights='imagenet', classes=4).to(app)
opt = torch.optim.AdamW(modele.parameters(), lr=LR)
planif = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOQUES)
scaler = torch.amp.GradScaler('cuda', enabled=app == 'cuda')
perte = nn.CrossEntropyLoss(weight=torch.tensor([0.3, 1.0, 4.0, 4.0]).to(app))  # portes et fenêtres rares : pesées davantage
debut_ep, meilleur_iou = 0, 0.0
if os.path.exists(REPRISE):
    ck = torch.load(REPRISE, map_location=app); modele.load_state_dict(ck['modele']); opt.load_state_dict(ck['opt']); planif.load_state_dict(ck['planif'])
    debut_ep = ck['epoque'] + 1; meilleur_iou = ck['meilleur_iou']; print(f'reprise à l\'époque {debut_ep + 1}/{EPOQUES}, meilleur mIoU {meilleur_iou:.3f}')
def iou_par_classe(pred, cible):
    return [((pred == c) & (cible == c)).sum().item() / max(1, ((pred == c) | (cible == c)).sum().item()) for c in range(4)]
for ep in range(debut_ep, EPOQUES):
    modele.train(); total = 0; t0 = time.time()
    for x, y in dl_train:
        x, y = x.to(app), y.to(app); opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=app, enabled=app == 'cuda'):
            l = perte(modele(x), y)
        scaler.scale(l).backward(); scaler.step(opt); scaler.update(); total += l.item()
    planif.step(); modele.eval(); ious = np.zeros(4); nb = 0
    with torch.no_grad():
        for x, y in dl_val:
            p = modele(x.to(app)).argmax(1).cpu(); ious += np.array(iou_par_classe(p, y)); nb += 1
    ious /= max(1, nb); miou = float(ious[1:].mean())
    print(f'époque {ep + 1}/{EPOQUES} — perte {total / max(1, len(dl_train)):.3f} — IoU fond/mur/porte/fenêtre {np.round(ious, 3)} — mIoU {miou:.3f} — {int(time.time() - t0)} s')
    torch.save({'modele': modele.state_dict(), 'opt': opt.state_dict(), 'planif': planif.state_dict(), 'epoque': ep, 'meilleur_iou': max(meilleur_iou, miou)}, REPRISE)
    if miou > meilleur_iou:
        meilleur_iou = miou; torch.save(modele.state_dict(), MEILLEUR); print('  → meilleur modèle enregistré')
print('entraînement terminé — meilleur mIoU (mur, porte, fenêtre) :', round(meilleur_iou, 3))

In [ ]:
# 6. Export ONNX (exportateur classique, poids inclus, vérifié) et téléchargement
import onnx
modele.load_state_dict(torch.load(MEILLEUR, map_location='cpu')); modele.eval().cpu()
exemple = torch.zeros(1, 3, TAILLE, TAILLE)
try:
    torch.onnx.export(modele, exemple, ONNX, input_names=['image'], output_names=['logits'], opset_version=17, dynamic_axes={'image': {0: 'lot'}, 'logits': {0: 'lot'}}, dynamo=False)
except TypeError:
    torch.onnx.export(modele, exemple, ONNX, input_names=['image'], output_names=['logits'], opset_version=17, dynamic_axes={'image': {0: 'lot'}, 'logits': {0: 'lot'}})
m = onnx.load(ONNX, load_external_data=True); onnx.save_model(m, ONNX, save_as_external_data=False); onnx.checker.check_model(ONNX)
taille_mo = os.path.getsize(ONNX) / 1_000_000
assert taille_mo > 10, 'Fichier ONNX anormalement petit : les poids ne sont pas dedans.'
print(f'ONNX écrit : {ONNX} — {taille_mo:.1f} Mo')
# Contrôle d\'inférence CPU
import onnxruntime as ort
sess = ort.InferenceSession(ONNX, providers=['CPUExecutionProvider'])
print('inférence ONNX OK, sortie', sess.run(None, {'image': np.zeros((1, 3, TAILLE, TAILLE), np.float32)})[0].shape)
try:
    from google.colab import files; files.download(ONNX)
    print('→ Fichier téléchargé : promopro-plan3d.onnx (aussi sur Drive, dossier promopro-ia). C\'est le fichier à me transmettre.')
except Exception as e:
    print('Téléchargement automatique impossible (', e, ') : récupérez promopro-plan3d.onnx dans Drive → promopro-ia.')